# LViT Kaggle From Scratch (README-based)

This notebook rebuilds the pipeline from scratch following the official LViT README workflow:
1. Prepare dataset in LViT folder format
2. Train with `python train_model.py`
3. Evaluate with `python test_model.py`

Source README: https://github.com/HUANGLIZI/LViT


In [ ]:
# 0) Main config
from pathlib import Path
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

CFG = {
    "repo_url": "https://github.com/HUANGLIZI/LViT.git",
    "repo_dir": "/kaggle/working/LViT",
    "dataset_root": "/kaggle/input",
    "dataset_path_hint": None,
    "task_name": "QaTa-Covid19",  # or QaTa-Covid19

    "model_name": "LViT",
    "epochs": 2,
    "learning_rate": 1e-3,
    "batch_size": 16,
    "img_size": 224,

    "num_workers": 2,
    "pin_memory": False,
    "use_two_gpus": False,
    "cuda_visible_devices": "0",
    "force_single_gpu": True,
    "batch_size_per_gpu": 16,
    "use_amp": False,
    "train_subset_ratio": 0.50,
    "drop_last": False,
    "lv_loss_weight": 0.2,
    "epi_loss_weight": 0.05,
    "epi_beta": 0.01,
    "pseudo_confidence_threshold": 0.55,
    "semi_supervised": True,
    "profile_model": False,
    "deterministic": True,

    "run_name": "QaTa_Covid19_LViT_50pct_labels",
    "print_frequency": 20,
    "vis_frequency": 9999,
    "tensorboard": False,
    "stdout_every_n_lines": 30,

    # Artifact/report settings
    "save_every_epoch": True,
    "save_every_n_epochs": 20,
    "max_epoch_checkpoints_to_keep": 2,
    "save_pt_each_epoch": True,
    "save_optimizer_in_last": True,
    "save_optimizer_in_best": False,
    "export_metrics_report": True,
    "keep_visualize_last_n_for_zip": 10,

    "use_train_text_for_val_if_missing": True,
    "install_requirements_txt": False,
    "force_reclone_repo": True,
}

print(CFG)


In [ ]:
# 1) Clone repo + install dependencies
import os
import subprocess
import shutil
from pathlib import Path

repo_dir = Path(CFG["repo_dir"])
repo_url = CFG["repo_url"]

if repo_dir.exists() and bool(CFG.get("force_reclone_repo", True)):
    print("Removing existing repo for clean patching:", repo_dir)
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    print("Cloning repo...")
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    print("Repo already exists:", repo_dir)

if bool(CFG.get("install_requirements_txt", False)):
    req = repo_dir / "requirements.txt"
    if req.exists():
        print("Installing requirements.txt...")
        subprocess.run(["pip", "-q", "install", "-r", str(req)], check=True)


# Ensure minimal runtime deps used by train_model.py
subprocess.run(["pip", "-q", "install", "tensorboardX", "thop"], check=False)


In [ ]:
# 2) Resolve dataset from Kaggle Input and link to repo/datasets/<task_name>
import os
import shutil
from pathlib import Path

repo_dir = Path(CFG["repo_dir"])
requested_task = str(CFG["task_name"]).strip()
req_key = requested_task.lower().replace(" ", "")

TASK_ALIASES = {
    "mosmed": ["MosMedDataPlus", "MosMedData+", "MosMedData_plus", "MosMedData"],
    "qata": ["QaTa-Covid19", "QaTa-COV19", "QaTa_COV19", "QaTa-Covid-19", "Qata-Covid19"],
}

if "mosmed" in req_key:
    candidate_names = TASK_ALIASES["mosmed"]
    canonical_task_name = "MosMedDataPlus"
elif "qata" in req_key:
    candidate_names = TASK_ALIASES["qata"]
    canonical_task_name = "QaTa-Covid19"
else:
    candidate_names = [requested_task]
    canonical_task_name = requested_task

SPLITS = ("Train_Folder", "Val_Folder", "Test_Folder")

def is_lvit_task_dir(task_dir: Path) -> bool:
    for s in SPLITS:
        if not (task_dir / s / "img").is_dir():
            return False
        if not (task_dir / s / "labelcol").is_dir():
            return False
    return True

def discover_ready_task_dirs(root: Path):
    found = []
    if not root.exists():
        return found

    seen = set()
    probes = [root]
    for nm in candidate_names:
        probes += [root / nm, root / "datasets" / nm]

    try:
        children = [p for p in root.iterdir() if p.is_dir()]
    except Exception:
        children = []

    for c in children:
        probes.append(c)
        for nm in candidate_names:
            probes += [c / nm, c / "datasets" / nm]

    for p in probes:
        sp = str(p)
        if sp in seen:
            continue
        seen.add(sp)
        if p.exists() and is_lvit_task_dir(p):
            found.append(p)

    try:
        for train_dir in root.rglob("Train_Folder"):
            if not train_dir.is_dir():
                continue
            task_dir = train_dir.parent
            sp = str(task_dir)
            if sp in seen:
                continue
            seen.add(sp)
            if is_lvit_task_dir(task_dir):
                found.append(task_dir)
    except Exception:
        pass

    return found

search_roots = []
if CFG.get("dataset_path_hint"):
    search_roots.append(Path(CFG["dataset_path_hint"]))
search_roots += [Path(CFG["dataset_root"]), Path("/kaggle/input"), Path("/kaggle/working/datasets")]

uniq = []
seen = set()
for r in search_roots:
    rs = str(r)
    if rs not in seen:
        uniq.append(r)
        seen.add(rs)

candidates = []
for r in uniq:
    candidates.extend(discover_ready_task_dirs(r))

if not candidates:
    visible = {}
    for r in uniq:
        if r.exists():
            try:
                visible[str(r)] = [p.name for p in r.iterdir() if p.is_dir()][:10]
            except Exception:
                visible[str(r)] = []
    raise AssertionError(
        f"Dataset not found for task='{requested_task}'. "
        f"Expected README format with Train/Val/Test_Folder + img/labelcol. "
        f"Search roots={list(map(str, uniq))}; visible={visible}"
    )

name_rank = {n.lower().replace(" ", ""): i for i, n in enumerate(candidate_names)}
def score(p: Path):
    pn = p.name.lower().replace(" ", "")
    exact = name_rank.get(pn, 999)
    semantic = 0
    if "mosmed" in req_key and "mosmed" not in pn:
        semantic = 1
    if "qata" in req_key and "qata" not in pn:
        semantic = 1
    return (exact, semantic, len(str(p)))

source_task_dir = sorted(candidates, key=score)[0]
print("Using source task dir:", source_task_dir)

train_text = source_task_dir / "Train_Folder" / "Train_text.xlsx"
val_text = source_task_dir / "Val_Folder" / "Val_text.xlsx"
test_text = source_task_dir / "Test_Folder" / "Test_text.xlsx"

if not train_text.exists():
    raise FileNotFoundError(f"Missing: {train_text}")
if not val_text.exists() and CFG.get("use_train_text_for_val_if_missing", True):
    shutil.copy2(train_text, val_text)
    print("Val_text.xlsx missing; copied from Train_text.xlsx")
if not test_text.exists():
    raise FileNotFoundError(f"Missing: {test_text}")

repo_datasets = repo_dir / "datasets"
repo_datasets.mkdir(parents=True, exist_ok=True)
linked_task_dir = repo_datasets / canonical_task_name

if linked_task_dir.exists() or linked_task_dir.is_symlink():
    if linked_task_dir.is_symlink() or linked_task_dir.is_file():
        linked_task_dir.unlink()
    else:
        shutil.rmtree(linked_task_dir)

try:
    os.symlink(str(source_task_dir), str(linked_task_dir), target_is_directory=True)
    print(f"Symlinked dataset: {linked_task_dir} -> {source_task_dir}")
except Exception:
    shutil.copytree(source_task_dir, linked_task_dir)
    print(f"Symlink unavailable, copied dataset to: {linked_task_dir}")


def count_files(d: Path):
    return len([p for p in d.iterdir() if p.is_file()]) if d.exists() else 0

for split in SPLITS:
    n_img = count_files(linked_task_dir / split / "img")
    n_msk = count_files(linked_task_dir / split / "labelcol")
    print(f"{split}: img={n_img}, labelcol={n_msk}")

TASK_NAME_CANONICAL = canonical_task_name
TASK_DIR = linked_task_dir
print("TASK_NAME_CANONICAL =", TASK_NAME_CANONICAL)
print("TASK_DIR =", TASK_DIR)


In [ ]:
# 3) Patch repo for Kaggle compatibility (checkpoint + dataset loader fixes)
from pathlib import Path

repo_dir = Path(CFG["repo_dir"])

def _resolve_repo_file(repo_root: Path, rel_name: str) -> Path:
    direct = repo_root / rel_name
    if direct.exists():
        return direct
    for p in repo_root.rglob(rel_name):
        if p.is_file():
            return p
    raise FileNotFoundError(f"Could not find {rel_name} under {repo_root}")

# ---- Patch train_model.py ----
train_model_path = _resolve_repo_file(repo_dir, "train_model.py")
tr = train_model_path.read_text(encoding="utf-8")
tr_orig = tr

if "import json" not in tr:
    if "import os\n" in tr:
        tr = tr.replace("import os\n", "import os\nimport json\n", 1)
    else:
        tr = "import json\n" + tr

# Layer 1 fix: ensure save_checkpoint always creates parent directory.
if "def save_checkpoint(state, save_path):" in tr and "os.makedirs(save_path, exist_ok=True)" not in tr:
    tr = tr.replace(
        "def save_checkpoint(state, save_path):\n",
        "def save_checkpoint(state, save_path):\n    os.makedirs(save_path, exist_ok=True)\n",
        1,
    )

# Layer 1 fix: ensure last checkpoint save cannot fail when model_path is missing.
needle_a = "torch.save(last_state, config.model_path + '/last_checkpoint.pth.tar')"
needle_b = 'torch.save(last_state, config.model_path + "/last_checkpoint.pth.tar")'
if "os.makedirs(config.model_path, exist_ok=True)" not in tr:
    if needle_a in tr:
        tr = tr.replace(
            needle_a,
            "os.makedirs(config.model_path, exist_ok=True)\n        " + needle_a,
            1,
        )
    elif needle_b in tr:
        tr = tr.replace(
            needle_b,
            "os.makedirs(config.model_path, exist_ok=True)\n        " + needle_b,
            1,
        )

# Layer 1.6 fix: support MosMedDataPlus / QaTa-Covid19 task names.
start = tr.find("    if config.task_name == 'MoNuSeg':")
if start == -1:
    start = tr.find("    task_name_norm = str(config.task_name).lower().replace(' ', '').replace('-', '').replace('_', '')")
end = tr.find("\n\n    train_loader = DataLoader(", start)
if start != -1 and end != -1:
    new_dataset_block = """    task_name_norm = str(config.task_name).lower().replace(' ', '').replace('-', '').replace('_', '')
    if task_name_norm in ('monuseg', 'mosmeddataplus', 'mosmeddata+', 'mosmeddata', 'qatacovid19', 'qatacov19'):
        train_text = read_text(config.train_dataset + 'Train_text.xlsx')
        val_text_path = config.val_dataset + 'Val_text.xlsx'
        if os.path.exists(val_text_path):
            val_text = read_text(val_text_path)
        else:
            val_text = train_text
        train_dataset = ImageToImage2D(config.train_dataset, config.task_name, train_text, train_tf,
                                       image_size=config.img_size)
        val_dataset = ImageToImage2D(config.val_dataset, config.task_name, val_text, val_tf, image_size=config.img_size)
    elif task_name_norm in ('covid19',):
        text = read_text(config.task_dataset + 'Train_Val_text.xlsx')
        train_dataset = ImageToImage2D(config.train_dataset, config.task_name, text, train_tf,
                                       image_size=config.img_size)
        val_dataset = ImageToImage2D(config.val_dataset, config.task_name, text, val_tf, image_size=config.img_size)
    else:
        raise ValueError(f'Unsupported task_name: {config.task_name}')

    # Semi-supervised split: keep full images/text each epoch, reduce only GT supervision ratio.
    train_subset_ratio = float(getattr(config, 'train_subset_ratio', 1.0))
    if train_subset_ratio <= 0.0:
        raise ValueError(f'train_subset_ratio must be > 0, got {train_subset_ratio}')
    if train_subset_ratio > 1.0:
        train_subset_ratio = 1.0

    full_train_len = len(train_dataset)
    labeled_count = max(1, int(round(full_train_len * train_subset_ratio)))
    labeled_count = min(full_train_len, labeled_count)

    subset_generator = torch.Generator()
    subset_generator.manual_seed(int(getattr(config, 'seed', 666)))
    labeled_indices = torch.randperm(full_train_len, generator=subset_generator)[:labeled_count].tolist()

    labeled_names = []
    if hasattr(train_dataset, 'images_list'):
        for idx in labeled_indices:
            try:
                labeled_names.append(str(train_dataset.images_list[idx]))
            except Exception:
                pass

    config.labeled_image_names = set(labeled_names)
    config.labeled_image_count = int(len(labeled_names))
    config.total_train_samples = int(full_train_len)

    if not os.path.isdir(config.save_path):
        os.makedirs(config.save_path, exist_ok=True)
    labeled_meta_path = os.path.join(config.save_path, 'labeled_subset_meta.json')
    labeled_meta = {
        'train_subset_ratio': float(train_subset_ratio),
        'labeled_count': int(len(labeled_names)),
        'total_train_samples': int(full_train_len),
        'seed': int(getattr(config, 'seed', 666)),
        'labeled_image_names': labeled_names,
    }
    with open(labeled_meta_path, 'w', encoding='utf-8') as f:
        json.dump(labeled_meta, f, ensure_ascii=False, indent=2)
    config.labeled_subset_meta_path = labeled_meta_path

    logging.info(
        f'Using labeled GT subset for supervised loss: {len(labeled_names)}/{full_train_len} ({train_subset_ratio:.0%}); '
        f'keeping full images/text for EPI+LV on unlabeled samples.'
    )
    print(
        f'Using labeled GT subset for supervised loss: {len(labeled_names)}/{full_train_len} ({train_subset_ratio:.0%}); '
        f'keeping full images/text for EPI+LV on unlabeled samples.'
    )
    print(f'Labeled subset metadata: {labeled_meta_path}')
"""
    tr = tr[:start] + new_dataset_block + tr[end:]

# Layer 1.9 fix: save best model from first epoch to avoid missing checkpoint on short runs.
tr = tr.replace("if epoch + 1 > 5:", "if epoch + 1 >= 1:")

# Layer 1.10 fix: always save last checkpoint every epoch for reliable eval fallback.
if "# save_last_checkpoint_every_epoch" not in tr:
    ckpt_anchor = "        early_stopping_count = epoch - best_epoch + 1"
    ckpt_block = """        # save_last_checkpoint_every_epoch\n        os.makedirs(config.model_path, exist_ok=True)\n        last_state = {\n            'epoch': epoch,\n            'best_model': False,\n            'model': model_type,\n            'state_dict': model.state_dict(),\n            'val_loss': val_loss,\n            'optimizer': optimizer.state_dict(),\n        }\n        torch.save(last_state, config.model_path + '/last_checkpoint.pth.tar')\n"""
    if ckpt_anchor in tr:
        tr = tr.replace(ckpt_anchor, ckpt_block + ckpt_anchor, 1)

# Keep tensorboard flag from Config.py instead of hard-coded True.
tr = tr.replace(
    "model = main_loop(model_type=config.model_name, tensorboard=True)",
    "model = main_loop(model_type=config.model_name, tensorboard=config.tensorboard)",
)

if tr != tr_orig:
    train_model_path.write_text(tr, encoding="utf-8")
    print("Patched:", train_model_path)
else:
    print("No patch needed:", train_model_path)

# ---- Patch Load_Dataset.py ----
# Layer 1.5 fix: make text embedding robust when `bert_embedding` is unavailable.
load_dataset_path = _resolve_repo_file(repo_dir, "Load_Dataset.py")
ld = load_dataset_path.read_text(encoding="utf-8")
ld_orig = ld

if "from bert_embedding import BertEmbedding" in ld and "BertEmbeddingFallback" not in ld:
    ld = ld.replace(
        "from bert_embedding import BertEmbedding",
        """try:
    from bert_embedding import BertEmbedding  # type: ignore
except Exception:
    import hashlib

    class BertEmbeddingFallback:
        def __init__(self, *args, **kwargs):
            self.dim = 768

        def _encode_one(self, token: str):
            seed = int(hashlib.md5(token.encode('utf-8', errors='ignore')).hexdigest()[:8], 16)
            rng = np.random.RandomState(seed)
            return rng.normal(loc=0.0, scale=0.02, size=(self.dim,)).astype(np.float32)

        def __call__(self, text):
            # Mimic bert_embedding output: list[(tokens, np.ndarray[num_tokens, 768])]
            # LViT expects >=10 tokens (ImageToImage2D) or >=14 (LV2D), so we pad to be safe.
            import re

            sentences = text if isinstance(text, (list, tuple)) else [str(text)]
            sentences = [str(s) for s in sentences if s is not None]
            merged = " ".join(sentences).strip()
            tokens = [t for t in re.findall(r"[A-Za-z0-9_]+", merged) if t]
            if len(tokens) == 0:
                tokens = ["empty"]

            min_tokens = 16
            if len(tokens) < min_tokens:
                tokens = tokens + [f"pad_{i}" for i in range(min_tokens - len(tokens))]

            embs = [self._encode_one(tok) for tok in tokens]
            return [(tokens, np.stack(embs, axis=0).astype(np.float32))]

    BertEmbedding = BertEmbeddingFallback
""",
        1,
    )


# Layer 1.7 fix: enforce fixed token length expected by LViT.
lv_old = """        text = np.array(text_token[0][1])
        if text.shape[0] > 14:
            text = text[:14, :]
"""
lv_new = """        text = np.array(text_token[0][1], dtype=np.float32)
        if text.ndim == 1:
            text = np.expand_dims(text, axis=0)
        if text.shape[0] < 14:
            pad = np.zeros((14 - text.shape[0], text.shape[1]), dtype=np.float32)
            text = np.concatenate([text, pad], axis=0)
        elif text.shape[0] > 14:
            text = text[:14, :]
"""

img_old = """        text = np.array(text_token[0][1])
        if text.shape[0] > 10:
            text = text[:10, :]
"""
img_new = """        text = np.array(text_token[0][1], dtype=np.float32)
        if text.ndim == 1:
            text = np.expand_dims(text, axis=0)
        if text.shape[0] < 10:
            pad = np.zeros((10 - text.shape[0], text.shape[1]), dtype=np.float32)
            text = np.concatenate([text, pad], axis=0)
        elif text.shape[0] > 10:
            text = text[:10, :]
"""

if lv_old in ld:
    ld = ld.replace(lv_old, lv_new, 1)
if img_old in ld:
    ld = ld.replace(img_old, img_new, 1)

if ld != ld_orig:
    load_dataset_path.write_text(ld, encoding="utf-8")
    print("Patched:", load_dataset_path)
else:
    print("No patch needed:", load_dataset_path)


# ---- Patch Train_one_epoch.py ----
train_one_epoch_path = _resolve_repo_file(repo_dir, "Train_one_epoch.py")
to = train_one_epoch_path.read_text(encoding="utf-8")
to_orig = to

if "import json" not in to:
    to = to.replace("import os\n", "import os\nimport json\n", 1)

if "import torch.nn.functional as F" not in to:
    to = to.replace("import torch.optim\n", "import torch.optim\nimport torch.nn.functional as F\n", 1)

helper_block = """

_LABELED_NAME_SET_CACHE = None


def _normalize_name(name):
    return os.path.basename(str(name)).strip()


def _get_labeled_name_set():
    global _LABELED_NAME_SET_CACHE
    if _LABELED_NAME_SET_CACHE is not None:
        return _LABELED_NAME_SET_CACHE

    name_set = set()

    raw_names = getattr(config, 'labeled_image_names', None)
    if raw_names is not None:
        try:
            name_set = {_normalize_name(x) for x in list(raw_names)}
        except Exception:
            name_set = set()

    if len(name_set) == 0:
        meta_path = getattr(config, 'labeled_subset_meta_path', None)
        if meta_path and os.path.exists(meta_path):
            try:
                with open(meta_path, 'r', encoding='utf-8') as f:
                    meta = json.load(f)
                name_set = {_normalize_name(x) for x in meta.get('labeled_image_names', [])}
            except Exception:
                name_set = set()

    _LABELED_NAME_SET_CACHE = name_set
    return _LABELED_NAME_SET_CACHE


def _pred_to_prob(preds):
    if torch.is_floating_point(preds):
        p_min = float(preds.detach().min().item())
        p_max = float(preds.detach().max().item())
        if p_min < 0.0 or p_max > 1.0:
            return torch.sigmoid(preds)
    return preds
"""

if "_get_labeled_name_set" not in to:
    insert_at = to.find("##################################################################################\n#=================================================================================\n#          Train One Epoch")
    if insert_at != -1:
        to = to[:insert_at] + helper_block + "\n" + to[insert_at:]

fn_start = to.find("def train_one_epoch(loader, model, criterion, optimizer, writer, epoch, lr_scheduler, model_type, logger):")
if fn_start == -1:
    raise RuntimeError("Could not find train_one_epoch function in Train_one_epoch.py")

fn_end_anchor = "    return average_loss, train_dice_avg"
fn_end = to.find(fn_end_anchor, fn_start)
if fn_end == -1:
    raise RuntimeError("Could not find train_one_epoch end anchor in Train_one_epoch.py")
fn_end = to.find("\n", fn_end + len(fn_end_anchor))
if fn_end == -1:
    fn_end = len(to)

new_train_one_epoch_fn = """def train_one_epoch(loader, model, criterion, optimizer, writer, epoch, lr_scheduler, model_type, logger):
    logging_mode = 'Train' if model.training else 'Val'
    end = time.time()
    time_sum, loss_sum = 0.0, 0.0
    sup_loss_sum, lv_loss_sum, epi_loss_sum = 0.0, 0.0, 0.0
    pseudo_update_sum = 0.0
    labeled_seen_sum, unlabeled_seen_sum = 0.0, 0.0
    dice_sum, iou_sum, acc_sum = 0.0, 0.0, 0.0

    labeled_name_set = _get_labeled_name_set() if model.training else set()
    subset_ratio = float(getattr(config, 'train_subset_ratio', 1.0))
    subset_ratio = min(max(subset_ratio, 0.0), 1.0)
    if model.training:
        logger.info(
            f"[SemiSup] subset_ratio={subset_ratio:.4f} labeled_name_count={len(labeled_name_set)}"
        )
        if subset_ratio < 0.999 and len(labeled_name_set) == 0:
            raise RuntimeError(
                'train_subset_ratio < 1 but labeled_name_set is empty. '
                'Pseudo-label branch cannot be activated; check labeled subset metadata.'
            )
    lv_w = float(getattr(config, 'lv_loss_weight', 0.2))
    epi_w = float(getattr(config, 'epi_loss_weight', 0.05))
    epi_beta = float(getattr(config, 'epi_beta', 0.01))
    epi_beta = min(max(epi_beta, 0.0), 0.9999)
    pseudo_thr = float(getattr(config, 'pseudo_confidence_threshold', 0.55))
    pseudo_thr = min(max(pseudo_thr, 0.5), 0.999)
    if not hasattr(train_one_epoch, '_epi_ema_cache'):
        train_one_epoch._epi_ema_cache = {}
    epi_ema_cache = train_one_epoch._epi_ema_cache

    for i, (sampled_batch, names) in enumerate(loader, 1):
        try:
            loss_name = criterion._get_name()
        except AttributeError:
            loss_name = criterion.__name__

        images, masks, text = sampled_batch['image'], sampled_batch['label'], sampled_batch['text']
        if text.shape[1] > 10:
            text = text[:, :10, :]

        images, masks, text = images.cuda(), masks.cuda(), text.cuda()
        masks_f = masks.float()

        preds = model(images, text)
        probs = _pred_to_prob(preds)

        batch_size = len(images)
        batch_names = [_normalize_name(nm) for nm in names]

        if model.training:
            if len(labeled_name_set) > 0:
                is_labeled_list = [nm in labeled_name_set for nm in batch_names]
                is_labeled = torch.tensor(is_labeled_list, dtype=torch.bool, device=images.device)
            else:
                is_labeled = torch.ones((batch_size,), dtype=torch.bool, device=images.device)
        else:
            is_labeled = torch.ones((batch_size,), dtype=torch.bool, device=images.device)

        unlabeled_mask = ~is_labeled
        labeled_seen_sum += float(is_labeled.sum().item())
        unlabeled_seen_sum += float(unlabeled_mask.sum().item())

        if int(is_labeled.sum().item()) > 0:
            sup_loss = criterion(probs[is_labeled], masks_f[is_labeled])
        else:
            sup_loss = probs.mean() * 0.0

        lv_loss = probs.mean() * 0.0
        epi_loss = probs.mean() * 0.0
        pseudo_update_count_step = 0.0

        if model.training and int(unlabeled_mask.sum().item()) > 0:
            unlabeled_probs = probs[unlabeled_mask]
            unlabeled_det = unlabeled_probs.detach()
            unlabeled_indices = torch.nonzero(unlabeled_mask, as_tuple=False).view(-1).tolist()
            unlabeled_names = [batch_names[j] for j in unlabeled_indices]
            ema_targets = []
            for uidx, nm in enumerate(unlabeled_names):
                current_pt = unlabeled_det[uidx]
                prev_pt = epi_ema_cache.get(nm)
                if prev_pt is None or tuple(prev_pt.shape) != tuple(current_pt.shape):
                    ema_pt = current_pt.clone()
                else:
                    ema_pt = epi_beta * prev_pt.to(current_pt.device) + (1.0 - epi_beta) * current_pt
                epi_ema_cache[nm] = ema_pt.detach().cpu()
                ema_targets.append(ema_pt)

            if len(ema_targets) > 0:
                ema_target_tensor = torch.stack(ema_targets, dim=0).to(unlabeled_probs.device)
                epi_loss = F.mse_loss(unlabeled_probs, ema_target_tensor)

            pseudo_targets = (unlabeled_det >= 0.5).float()
            conf_mask = ((unlabeled_det >= pseudo_thr) | (unlabeled_det <= (1.0 - pseudo_thr))).float()
            conf_count = float(conf_mask.sum().item())
            pseudo_update_count_step = conf_count

            if conf_count > 0:
                lv_map = F.binary_cross_entropy(unlabeled_probs, pseudo_targets, reduction='none')
                lv_loss = (lv_map * conf_mask).sum() / (conf_count + 1e-6)

        out_loss = sup_loss + lv_w * lv_loss + epi_w * epi_loss

        if model.training:
            optimizer.zero_grad()
            out_loss.backward()
            optimizer.step()

        metric_probs = probs.detach().clone()
        train_dice = criterion._show_dice(metric_probs, masks_f.detach().clone())
        train_iou = iou_on_batch(masks, metric_probs)

        batch_time = time.time() - end
        if epoch % config.vis_frequency == 0 and logging_mode == 'Val':
            vis_path = config.visualize_path + str(epoch) + '/'
            if not os.path.isdir(vis_path):
                os.makedirs(vis_path)
            save_on_batch(images, masks, metric_probs, names, vis_path)

        time_sum += batch_size * float(batch_time)
        loss_sum += batch_size * float(out_loss.detach().item())
        sup_loss_sum += batch_size * float(sup_loss.detach().item())
        lv_loss_sum += batch_size * float(lv_loss.detach().item())
        epi_loss_sum += batch_size * float(epi_loss.detach().item())
        pseudo_update_sum += float(pseudo_update_count_step)

        iou_sum += batch_size * float(train_iou)
        dice_sum += batch_size * float(train_dice)

        if i == len(loader):
            denom = config.batch_size * (i - 1) + batch_size
        else:
            denom = i * config.batch_size

        average_loss = loss_sum / max(1.0, float(denom))
        average_sup_loss = sup_loss_sum / max(1.0, float(denom))
        average_lv_loss = lv_loss_sum / max(1.0, float(denom))
        average_epi_loss = epi_loss_sum / max(1.0, float(denom))

        average_time = time_sum / max(1.0, float(denom))
        train_iou_average = iou_sum / max(1.0, float(denom))
        train_dice_avg = dice_sum / max(1.0, float(denom))

        end = time.time()
        torch.cuda.empty_cache()

        if i % config.print_frequency == 0:
            print_summary(epoch + 1, i, len(loader), out_loss, loss_name, batch_time,
                          average_loss, average_time, train_iou, train_iou_average,
                          train_dice, train_dice_avg, 0, 0, logging_mode,
                          lr=min(g['lr'] for g in optimizer.param_groups), logger=logger)

            logger.info(
                f"   [{logging_mode}] sup_loss:{float(sup_loss.detach().item()):.6f} "
                f"lv_loss:{float(lv_loss.detach().item()):.6f} "
                f"epi_loss:{float(epi_loss.detach().item()):.6f} "
                f"total_loss:{float(out_loss.detach().item()):.6f} "
                f"pseudo_update_count:{int(pseudo_update_count_step)}"
            )

        if config.tensorboard:
            step = epoch * len(loader) + i
            writer.add_scalar(logging_mode + '_' + loss_name, out_loss.item(), step)
            writer.add_scalar(logging_mode + '_sup_loss', float(sup_loss.detach().item()), step)
            writer.add_scalar(logging_mode + '_lv_loss', float(lv_loss.detach().item()), step)
            writer.add_scalar(logging_mode + '_epi_loss', float(epi_loss.detach().item()), step)
            writer.add_scalar(logging_mode + '_pseudo_update_count', float(pseudo_update_count_step), step)
            writer.add_scalar(logging_mode + '_iou', train_iou, step)
            writer.add_scalar(logging_mode + '_dice', train_dice, step)

        torch.cuda.empty_cache()

    logger.info(
        f"[{logging_mode}-Epoch-Metrics] epoch={epoch + 1} "
        f"sup_loss:{average_sup_loss:.6f} "
        f"lv_loss:{average_lv_loss:.6f} "
        f"epi_loss:{average_epi_loss:.6f} "
        f"total_loss:{average_loss:.6f} "
        f"pseudo_update_count:{int(pseudo_update_sum)} "
        f"labeled_seen:{int(labeled_seen_sum)} "
        f"unlabeled_seen:{int(unlabeled_seen_sum)} "
        f"Dice:{train_dice_avg:.6f} IoU:{train_iou_average:.6f}"
    )

    if model.training and subset_ratio < 0.999 and unlabeled_seen_sum <= 0:
        raise RuntimeError(
            f'Expected unlabeled samples with train_subset_ratio={subset_ratio}, '
            'but unlabeled_seen==0 in this epoch. Check name matching or subset metadata.'
        )

    if lr_scheduler is not None:
        lr_scheduler.step()

    return average_loss, train_dice_avg
"""

to = to[:fn_start] + new_train_one_epoch_fn + to[fn_end:]

to = to.replace("logging_mode is 'Val'", "logging_mode == 'Val'")

if to != to_orig:
    train_one_epoch_path.write_text(to, encoding="utf-8")
    print("Patched:", train_one_epoch_path)
else:
    print("No patch needed:", train_one_epoch_path)


# ---- Patch test_model.py ----
test_model_path = _resolve_repo_file(repo_dir, "test_model.py")
te = test_model_path.read_text(encoding="utf-8")
te_orig = te

# Keep launcher CUDA mask; do not force to GPU0 inside script.
te = te.replace(
    '    os.environ["CUDA_VISIBLE_DEVICES"] = "0"',
    '    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0")'
)

# Support generic task names (MosMedDataPlus / QaTa-Covid19 / etc.)
old_task_block = """    if config.task_name == "MoNuSeg":
        test_num = 14
        model_type = config.model_name
        model_path = "./MoNuSeg/" + model_type + "/" + test_session + "/models/best_model-" + model_type + ".pth.tar"

    elif config.task_name == "Covid19":
        test_num = 2113
        model_type = config.model_name
        model_path = "./Covid19/" + model_type + "/" + test_session + "/models/best_model-" + model_type + ".pth.tar"
    
"""
new_task_block = """    model_type = config.model_name
    model_path = f"./{config.task_name}/{model_type}/{test_session}/models/best_model-{model_type}.pth.tar"
    if not os.path.exists(model_path):
        alt_last = f"./{config.task_name}/{model_type}/{test_session}/models/last_checkpoint.pth.tar"
        if os.path.exists(alt_last):
            model_path = alt_last
"""
if old_task_block in te:
    te = te.replace(old_task_block, new_task_block, 1)
elif 'model_type = config.model_name' not in te:
    te = te.replace(
        "    test_session = config.test_session\\n",
        "    test_session = config.test_session\\n    model_type = config.model_name\\n",
        1,
    )
# Use explicit torch.nn.DataParallel to avoid missing nn import edge cases.
te = te.replace('       model = nn.DataParallel(model)', '       model = torch.nn.DataParallel(model)')

# Resolve test text xlsx robustly.
old_test_text = """    test_text = read_text(config.test_dataset + 'Test_text.xlsx')
"""
new_test_text = """    test_text_path = os.path.join(config.test_dataset, 'Test_text.xlsx')
    if not os.path.exists(test_text_path):
        for nm in ['test_text.xlsx', 'Val_text.xlsx', 'val_text.xlsx', 'Train_text.xlsx', 'train_text.xlsx', 'Train_Val_text.xlsx']:
            cand = os.path.join(config.test_dataset, nm)
            if os.path.exists(cand):
                test_text_path = cand
                break
    if not os.path.exists(test_text_path):
        raise FileNotFoundError(f'No text xlsx found under {config.test_dataset}')
    test_text = read_text(test_text_path)
"""
if old_test_text in te:
    te = te.replace(old_test_text, new_test_text, 1)

# Dynamic test set size.
if 'test_num = len(test_dataset)' not in te:
    te = te.replace(
        "    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)",
        "    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)\n    test_num = len(test_dataset)",
        1,
    )


# Fallback checkpoint discovery + robust state_dict loading.
old_ckpt_line = "    checkpoint = torch.load(model_path, map_location='cuda')\n"
new_ckpt_block = """    def _collect_ckpt_candidates(_model_dir):
        out = []
        if not os.path.isdir(_model_dir):
            return out

        primary = [
            os.path.join(_model_dir, f"best_model-{model_type}.pth.tar"),
            os.path.join(_model_dir, f"best_model-{model_type}.pt"),
            os.path.join(_model_dir, "last_checkpoint.pth.tar"),
            os.path.join(_model_dir, "last_checkpoint.pt"),
        ]
        for p in primary:
            if os.path.exists(p):
                out.append(p)

        extra = sorted([
            os.path.join(_model_dir, n)
            for n in os.listdir(_model_dir)
            if n.startswith("model-") and (n.endswith(".pth.tar") or n.endswith(".pt"))
        ])
        out.extend(extra)
        return out

    if not os.path.exists(model_path):
        model_dir = os.path.dirname(model_path)
        candidates = _collect_ckpt_candidates(model_dir)

        # Fallback to sibling runs under ./<task>/<model>/<run>/models
        model_root = os.path.dirname(model_dir)
        sibling_runs = []
        if os.path.isdir(model_root):
            for run_name in os.listdir(model_root):
                sib_dir = os.path.join(model_root, run_name, "models")
                if os.path.isdir(sib_dir):
                    sibling_runs.append((run_name, sib_dir))

            sibling_runs = sorted(
                sibling_runs,
                key=lambda x: os.path.getmtime(x[1]),
                reverse=True,
            )

            for _, sib_dir in sibling_runs:
                if os.path.abspath(sib_dir) == os.path.abspath(model_dir):
                    continue
                candidates.extend(_collect_ckpt_candidates(sib_dir))

        fallback = next((c for c in candidates if os.path.exists(c)), None)
        if fallback is None:
            available_runs = [rn for rn, _ in sibling_runs][:20]
            raise FileNotFoundError(
                f"No checkpoint found for requested run under {model_dir}. "
                f"Also checked sibling runs under {model_root}. "
                f"Available runs (up to 20): {available_runs}"
            )

        model_path = fallback
        print("Using fallback checkpoint:", model_path)

    map_loc = 'cuda' if torch.cuda.is_available() else 'cpu'
    checkpoint = torch.load(model_path, map_location=map_loc)
    state_dict = checkpoint['state_dict'] if isinstance(checkpoint, dict) and 'state_dict' in checkpoint else checkpoint
"""
if old_ckpt_line in te:
    te = te.replace(old_ckpt_line, new_ckpt_block, 1)

te = te.replace("    model.load_state_dict(checkpoint['state_dict'], strict=False)\n", "    model.load_state_dict(state_dict, strict=False)\n")

# Safe division when empty.
te = te.replace('print("dice_pred", dice_pred / test_num)', 'print("dice_pred", dice_pred / max(1, test_num))')
te = te.replace('print("iou_pred", iou_pred / test_num)', 'print("iou_pred", iou_pred / max(1, test_num))')

if te != te_orig:
    test_model_path.write_text(te, encoding="utf-8")
    print("Patched:", test_model_path)
else:
    print("No patch needed:", test_model_path)



In [ ]:
# 4) Write Config.py overrides for this run
from pathlib import Path
import os
import subprocess

repo_dir = Path(CFG["repo_dir"])
config_path = repo_dir / "Config.py"
text = config_path.read_text(encoding="utf-8")

start_marker = "# ==== Kaggle Override Start ===="
end_marker = "# ==== Kaggle Override End ===="

if start_marker in text and end_marker in text:
    text = text.split(start_marker)[0].rstrip() + "\n\n"

requested_visible = str(CFG.get("cuda_visible_devices", "0"))
use_two_gpus = bool(CFG.get("use_two_gpus", False))


def _probe_cuda_count(visible_devices: str):
    """Probe how many CUDA devices are fully usable in a fresh process."""
    env = os.environ.copy()
    if visible_devices == "":
        env.pop("CUDA_VISIBLE_DEVICES", None)
    else:
        env["CUDA_VISIBLE_DEVICES"] = visible_devices

    probe_code = """
import sys
import torch

usable = 0
if torch.cuda.is_available():
    total = torch.cuda.device_count()
    for idx in range(total):
        try:
            torch.cuda.set_device(idx)
            torch.cuda.get_device_properties(idx)
            torch.cuda.get_device_capability(idx)
            t = torch.zeros((1,), device=f"cuda:{idx}")
            _ = float(t.sum().item())
            torch.cuda.synchronize(idx)
            usable += 1
        except Exception as exc:
            sys.stderr.write(f'GPU_PROBE_FAIL idx={idx}: {exc}\\n')
            break

sys.stdout.write(str(usable))
""".strip()

    cmd = ["python", "-c", probe_code]
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=env)
    if p.returncode != 0:
        return 0, (p.stderr or "").strip()

    out = (p.stdout or "").strip()
    try:
        return int(out), ""
    except Exception:
        return 0, f"Unexpected probe output: {out!r}"


def _choose_visible_devices(prefer_two_gpu: bool, requested: str):
    # Candidate order: requested -> single GPU -> CPU fallback
    candidates = []
    for c in [requested, "0", ""]:
        if c not in candidates:
            candidates.append(c)

    probe_log = []
    for cand in candidates:
        cnt, err = _probe_cuda_count(cand)
        probe_log.append((cand, cnt, err))

        if cand == "":
            # CPU/no explicit mask fallback
            if cnt == 0:
                return cand, cnt, probe_log

        elif "," in cand:
            # Multi-GPU mask candidate
            if prefer_two_gpu and cnt >= 2:
                return cand, cnt, probe_log
            if (not prefer_two_gpu) and cnt >= 1:
                return cand, cnt, probe_log

        else:
            # Single GPU candidate
            if cnt >= 1:
                return cand, cnt, probe_log

    return "", 0, probe_log


effective_visible, gpu_count, probe_log = _choose_visible_devices(use_two_gpus, requested_visible)

if gpu_count >= 2 and use_two_gpus:
    effective_force_single = False
    effective_batch = int(max(1, CFG.get("batch_size_per_gpu", 2))) * gpu_count
elif gpu_count >= 1:
    effective_force_single = True
    effective_batch = int(max(1, CFG.get("batch_size_per_gpu", CFG.get("batch_size", 2))))
else:
    effective_force_single = True
    effective_batch = int(max(1, CFG.get("batch_size", 2)))

# Export to current notebook env so following subprocess calls inherit consistent GPU mask.
if effective_visible == "":
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = effective_visible

CFG["effective_cuda_visible_devices"] = effective_visible
CFG["effective_gpu_count"] = int(gpu_count)
CFG["effective_batch_size"] = int(effective_batch)
CFG["effective_force_single_gpu"] = bool(effective_force_single)

override = f"""
{start_marker}
save_model = True
tensorboard = {bool(CFG['tensorboard'])}
os.environ["CUDA_VISIBLE_DEVICES"] = "{effective_visible}"
use_cuda = torch.cuda.is_available()
seed = 666
os.environ['PYTHONHASHSEED'] = str(seed)

cosineLR = True
n_channels = 3
n_labels = 1
epochs = {int(CFG['epochs'])}
img_size = {int(CFG['img_size'])}
print_frequency = {int(CFG['print_frequency'])}
save_frequency = 5000
vis_frequency = {int(CFG['vis_frequency'])}
early_stopping_patience = 100000

pretrain = False
task_name = '{TASK_NAME_CANONICAL}'
learning_rate = {float(CFG['learning_rate'])}
batch_size = {int(effective_batch)}
model_name = '{CFG['model_name']}'

num_workers = {int(CFG['num_workers'])}
pin_memory = {bool(CFG['pin_memory'])}
use_amp = {bool(CFG.get('use_amp', True))}
train_subset_ratio = {float(CFG.get('train_subset_ratio', 1.0))}
drop_last = {bool(CFG.get('drop_last', False))}
profile_model = {bool(CFG.get('profile_model', False))}
semi_supervised = {bool(CFG.get('semi_supervised', True))}
lv_loss_weight = {float(CFG.get('lv_loss_weight', 0.2))}
epi_loss_weight = {float(CFG.get('epi_loss_weight', 0.05))}
epi_beta = {float(CFG.get('epi_beta', 0.01))}
pseudo_confidence_threshold = {float(CFG.get('pseudo_confidence_threshold', 0.7))}
force_single_gpu = {bool(effective_force_single)}
save_every_epoch = {bool(CFG['save_every_epoch'])}
save_pt_each_epoch = {bool(CFG['save_pt_each_epoch'])}
save_every_n_epochs = {int(CFG['save_every_n_epochs'])}
max_epoch_checkpoints_to_keep = {int(CFG['max_epoch_checkpoints_to_keep'])}
save_optimizer_in_last = {bool(CFG['save_optimizer_in_last'])}
save_optimizer_in_best = {bool(CFG['save_optimizer_in_best'])}

task_dataset = './datasets/' + task_name + '/Train_Folder/'
train_dataset = './datasets/' + task_name + '/Train_Folder/'
val_dataset = './datasets/' + task_name + '/Val_Folder/'
test_dataset = './datasets/' + task_name + '/Test_Folder/'

session_name = '{CFG['run_name']}'
save_path = task_name + '/' + model_name + '/' + session_name + '/'
model_path = save_path + 'models/'
tensorboard_folder = save_path + 'tensorboard_logs/'
logger_path = save_path + session_name + '.log'
visualize_path = save_path + 'visualize_val/'

test_session = session_name
{end_marker}
"""

text = text.rstrip() + "\n\n" + override
config_path.write_text(text, encoding="utf-8")

print("Config.py override written.")
print("task_name:", TASK_NAME_CANONICAL)
print("requested use_two_gpus:", use_two_gpus)
print("requested CUDA_VISIBLE_DEVICES:", requested_visible)
print("Probe results (candidate, usable_gpu_count, err):")
for cand, cnt, err in probe_log:
    print(" -", repr(cand), cnt, (err[:120] + "...") if err and len(err) > 120 else err)
if use_two_gpus and gpu_count < 2:
    print("[WARN] Runtime does not expose 2 usable GPUs. Falling back to safer mode.")
print("CUDA visible devices:", repr(effective_visible))
print("force_single_gpu:", effective_force_single)
print("effective batch_size:", effective_batch)
print("train_dataset:", f"./datasets/{TASK_NAME_CANONICAL}/Train_Folder/")



In [ ]:
# 5) Train (official README command)
import os
import subprocess
from pathlib import Path

repo_dir = Path(CFG["repo_dir"])
cmd = ["python", "train_model.py"]
print("$", " ".join(cmd))

# Layer 2 fix: create output dirs from Config.py right before train.
prep = subprocess.run(
    [
        "python",
        "-c",
        "import os,sys; sys.path.insert(0, '.'); import Config as c; "
        "os.makedirs(c.save_path, exist_ok=True); "
        "os.makedirs(c.model_path, exist_ok=True); "
        "os.makedirs(c.visualize_path, exist_ok=True)",
    ],
    cwd=repo_dir,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
if prep.returncode != 0:
    print("[WARN] Could not pre-create output dirs from Config.py")
    if prep.stdout:
        print(prep.stdout[-1500:])

env = os.environ.copy()
visible = str(CFG.get("effective_cuda_visible_devices", CFG.get("cuda_visible_devices", "0")))
if visible == "":
    env.pop("CUDA_VISIBLE_DEVICES", None)
else:
    env["CUDA_VISIBLE_DEVICES"] = visible
# Safety fallback: validate that all requested GPUs are truly usable.
probe_code = """
import sys
import torch

usable = 0
if torch.cuda.is_available():
    total = torch.cuda.device_count()
    for idx in range(total):
        try:
            torch.cuda.set_device(idx)
            torch.cuda.get_device_properties(idx)
            torch.cuda.get_device_capability(idx)
            t = torch.zeros((1,), device=f"cuda:{idx}")
            _ = float(t.sum().item())
            torch.cuda.synchronize(idx)
            usable += 1
        except Exception as exc:
            sys.stderr.write(f'GPU_PROBE_FAIL idx={idx}: {exc}\\n')
            break

print(usable)
""".strip()

probe = subprocess.run(
    ["python", "-c", probe_code],
    cwd=repo_dir,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
try:
    visible_count = int((probe.stdout or "0").strip().splitlines()[-1])
except Exception:
    visible_count = 0

if "," in visible and visible_count < 2:
    env["CUDA_VISIBLE_DEVICES"] = "0"
    visible = "0"
    print("[WARN] Multi-GPU mask is not fully usable in this runtime; fallback to single GPU: '0'")
print("CUDA_VISIBLE_DEVICES used for train:", repr(env.get("CUDA_VISIBLE_DEVICES", "<unset>")))

log_path = repo_dir / "train_live.log"
with log_path.open("w", encoding="utf-8") as lf:
    proc = subprocess.Popen(
        cmd,
        cwd=repo_dir,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        lf.write(line)
    rc = proc.wait()

if rc != 0:
    raise RuntimeError(f"Training failed with exit code {rc}. Full log: {log_path}")

models_dir = repo_dir / TASK_NAME_CANONICAL / CFG["model_name"] / CFG["run_name"] / "models"
ckpts = []
if models_dir.exists():
    ckpts = sorted(list(models_dir.glob("*.pth.tar")) + list(models_dir.glob("*.pt")))
print("models_dir:", models_dir)
print("checkpoint_count:", len(ckpts))
for cp in ckpts[:20]:
    print(" -", cp.name)
if len(ckpts) == 0:
    raise RuntimeError(
        f"Training finished but no checkpoint files were found in {models_dir}. "
        "Re-run cell #3 patch, then train again for at least 1 epoch."
    )
print("Training done.")
print(f"Live log saved: {log_path}")



In [ ]:

# 8) Epoch training log report: sup_loss/lv_loss/total_loss/pseudo_update_count + Dice/IoU
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd

repo_dir = Path(CFG["repo_dir"])
run_dir = repo_dir / TASK_NAME_CANONICAL / CFG["model_name"] / CFG["run_name"]

log_candidates = [
    repo_dir / "train_live.log",
    run_dir / f"{CFG['run_name']}.log",
]
log_path = None
for cand in log_candidates:
    if cand.exists() and cand.is_file():
        log_path = cand
        break

if log_path is None:
    raise FileNotFoundError(
        f"No training log found. Checked: {[str(p) for p in log_candidates]}"
    )

lines = log_path.read_text(encoding="utf-8", errors="ignore").splitlines()
print("Using log:", log_path)
print("Log lines:", len(lines))

epoch_header_re = re.compile(r"Epoch\s*\[(\d+)\s*/\s*(\d+)\]", re.I)
train_re = re.compile(
    r"\[Train\].*?Epoch:\s*\[(\d+)\].*?Loss:([0-9.eE+-]+).*?\(Avg\s*([0-9.eE+-]+)\).*?"
    r"IoU:([0-9.eE+-]+).*?\(Avg\s*([0-9.eE+-]+)\).*?"
    r"Dice:([0-9.eE+-]+).*?\(Avg\s*([0-9.eE+-]+)\)",
    re.I,
)
val_re = re.compile(
    r"\[Val\].*?Epoch:\s*\[(\d+)\].*?Loss:([0-9.eE+-]+).*?\(Avg\s*([0-9.eE+-]+)\).*?"
    r"IoU:([0-9.eE+-]+).*?\(Avg\s*([0-9.eE+-]+)\).*?"
    r"Dice:([0-9.eE+-]+).*?\(Avg\s*([0-9.eE+-]+)\)",
    re.I,
)

sup_loss_re = re.compile(r"\bsup[_\s-]*loss\b\s*[:=]\s*([0-9.eE+-]+)", re.I)
lv_loss_re = re.compile(r"\blv[_\s-]*loss\b\s*[:=]\s*([0-9.eE+-]+)", re.I)
epi_loss_re = re.compile(r"\bepi[_\s-]*loss\b\s*[:=]\s*([0-9.eE+-]+)", re.I)
total_loss_re = re.compile(r"\btotal[_\s-]*loss\b\s*[:=]\s*([0-9.eE+-]+)", re.I)
pseudo_cnt_re = re.compile(r"\bpseudo[_\s-]*update[_\s-]*count\b\s*[:=]\s*([0-9.eE+-]+)", re.I)
labeled_seen_re = re.compile(r"\blabeled[_\s-]*seen\b\s*[:=]\s*([0-9.eE+-]+)", re.I)
unlabeled_seen_re = re.compile(r"\bunlabeled[_\s-]*seen\b\s*[:=]\s*([0-9.eE+-]+)", re.I)


def _to_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


epochs = {}
current_epoch = None
max_epoch = None


def _ensure_epoch(ep):
    if ep not in epochs:
        epochs[ep] = {
            "epoch": int(ep),
            "sup_loss": np.nan,
            "lv_loss": np.nan,
            "epi_loss": np.nan,
            "total_loss": np.nan,
            "pseudo_update_count": np.nan,
            "labeled_seen": np.nan,
            "unlabeled_seen": np.nan,
            "train_loss_avg": np.nan,
            "train_dice_avg": np.nan,
            "train_iou_avg": np.nan,
            "val_loss_avg": np.nan,
            "val_dice_avg": np.nan,
            "val_iou_avg": np.nan,
            "sup_loss_found": False,
            "lv_loss_found": False,
            "epi_loss_found": False,
            "total_loss_found": False,
            "pseudo_update_count_found": False,
        }
    return epochs[ep]


for line in lines:
    m_head = epoch_header_re.search(line)
    if m_head:
        current_epoch = int(m_head.group(1))
        max_epoch = int(m_head.group(2))
        _ensure_epoch(current_epoch)

    m_train = train_re.search(line)
    if m_train:
        ep = int(m_train.group(1))
        rec = _ensure_epoch(ep)
        rec["train_loss_avg"] = _to_float(m_train.group(3))
        rec["train_iou_avg"] = _to_float(m_train.group(5))
        rec["train_dice_avg"] = _to_float(m_train.group(7))
        current_epoch = ep

    m_val = val_re.search(line)
    if m_val:
        ep = int(m_val.group(1))
        rec = _ensure_epoch(ep)
        rec["val_loss_avg"] = _to_float(m_val.group(3))
        rec["val_iou_avg"] = _to_float(m_val.group(5))
        rec["val_dice_avg"] = _to_float(m_val.group(7))
        current_epoch = ep

    target_ep = current_epoch
    if target_ep is None:
        continue
    rec = _ensure_epoch(target_ep)

    # Keep semi-supervised losses from Train logs only.
    is_train_line = ("[Train]" in line) or ("[Train-Epoch-Metrics]" in line)
    if not is_train_line:
        continue

    m = sup_loss_re.search(line)
    if m:
        rec["sup_loss"] = _to_float(m.group(1))
        rec["sup_loss_found"] = True

    m = lv_loss_re.search(line)
    if m:
        rec["lv_loss"] = _to_float(m.group(1))
        rec["lv_loss_found"] = True

    m = epi_loss_re.search(line)
    if m:
        rec["epi_loss"] = _to_float(m.group(1))
        rec["epi_loss_found"] = True

    m = total_loss_re.search(line)
    if m:
        rec["total_loss"] = _to_float(m.group(1))
        rec["total_loss_found"] = True

    m = pseudo_cnt_re.search(line)
    if m:
        rec["pseudo_update_count"] = _to_float(m.group(1))
        rec["pseudo_update_count_found"] = True

    m = labeled_seen_re.search(line)
    if m:
        rec["labeled_seen"] = _to_float(m.group(1))

    m = unlabeled_seen_re.search(line)
    if m:
        rec["unlabeled_seen"] = _to_float(m.group(1))

if not epochs:
    raise RuntimeError("No epoch information parsed from training log.")

rows = [epochs[k] for k in sorted(epochs.keys())]
df = pd.DataFrame(rows)

if max_epoch is not None:
    df["max_epoch"] = int(max_epoch)
else:
    df["max_epoch"] = np.nan

# Ensure expected columns always exist even when a metric is never printed in this run.
for col, default in {
    "sup_loss": np.nan,
    "lv_loss": np.nan,
    "epi_loss": np.nan,
    "total_loss": np.nan,
    "pseudo_update_count": np.nan,
    "labeled_seen": np.nan,
    "unlabeled_seen": np.nan,
    "sup_loss_found": False,
    "lv_loss_found": False,
    "epi_loss_found": False,
    "total_loss_found": False,
    "pseudo_update_count_found": False,
}.items():
    if col not in df.columns:
        df[col] = default

# Fill defaults when specific terms are not printed by the current training code.
df["total_loss"] = df["total_loss"].where(df["total_loss"].notna(), df["train_loss_avg"])
df["sup_loss"] = df["sup_loss"].where(df["sup_loss"].notna(), df["total_loss"])
df["lv_loss"] = df["lv_loss"].where(df["lv_loss"].notna(), 0.0)
df["epi_loss"] = df["epi_loss"].where(df["epi_loss"].notna(), 0.0)
df["pseudo_update_count"] = df["pseudo_update_count"].where(df["pseudo_update_count"].notna(), 0.0)
df["labeled_seen"] = df["labeled_seen"].where(df["labeled_seen"].notna(), 0.0)
df["unlabeled_seen"] = df["unlabeled_seen"].where(df["unlabeled_seen"].notna(), 0.0)

# Friendly aliases for report tables.
df["Dice"] = df["val_dice_avg"].where(df["val_dice_avg"].notna(), df["train_dice_avg"])
df["IoU"] = df["val_iou_avg"].where(df["val_iou_avg"].notna(), df["train_iou_avg"])

metrics_cols = [
    "epoch",
    "sup_loss",
    "lv_loss",
    "epi_loss",
    "total_loss",
    "pseudo_update_count",
    "labeled_seen",
    "unlabeled_seen",
    "Dice",
    "IoU",
    "train_loss_avg",
    "train_dice_avg",
    "train_iou_avg",
    "val_loss_avg",
    "val_dice_avg",
    "val_iou_avg",
    "sup_loss_found",
    "lv_loss_found",
    "epi_loss_found",
    "total_loss_found",
    "pseudo_update_count_found",
    "max_epoch",
]
df = df[metrics_cols]

epoch_csv = run_dir / "epoch_training_metrics.csv"
epoch_json = run_dir / "epoch_training_metrics.json"

df.to_csv(epoch_csv, index=False)
epoch_json.write_text(df.to_json(orient="records", indent=2), encoding="utf-8")

print("Saved:", epoch_csv)
print("Saved:", epoch_json)
print("Parsed epochs:", len(df))
print("Found sup_loss in log:", bool(df["sup_loss_found"].any()))
print("Found lv_loss in log:", bool(df["lv_loss_found"].any()))
print("Found epi_loss in log:", bool(df["epi_loss_found"].any()))
print("Found pseudo_update_count in log:", bool(df["pseudo_update_count_found"].any()))
display(df.tail(10))


In [ ]:
# 6) Evaluate (official README command)
import subprocess
from pathlib import Path
import os

repo_dir = Path(CFG["repo_dir"])
cmd = ["python", "test_model.py"]
print("$", " ".join(cmd))


def _probe_cuda_count_for_env(visible_devices: str):
    env = os.environ.copy()
    if visible_devices == "":
        env.pop("CUDA_VISIBLE_DEVICES", None)
    else:
        env["CUDA_VISIBLE_DEVICES"] = visible_devices

    probe_code = """
import sys
import torch

usable = 0
if torch.cuda.is_available():
    total = torch.cuda.device_count()
    for idx in range(total):
        try:
            torch.cuda.set_device(idx)
            torch.cuda.get_device_properties(idx)
            torch.cuda.get_device_capability(idx)
            t = torch.zeros((1,), device=f"cuda:{idx}")
            _ = float(t.sum().item())
            torch.cuda.synchronize(idx)
            usable += 1
        except Exception as exc:
            sys.stderr.write(f'GPU_PROBE_FAIL idx={idx}: {exc}\\n')
            break

sys.stdout.write(str(usable))
""".strip()

    p = subprocess.run(
        ["python", "-c", probe_code],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
    )
    if p.returncode != 0:
        return 0
    try:
        return int((p.stdout or "0").strip())
    except Exception:
        return 0


def _probe_cuda_smoke_for_env(visible_devices: str):
    env = os.environ.copy()
    if visible_devices == "":
        env.pop("CUDA_VISIBLE_DEVICES", None)
    else:
        env["CUDA_VISIBLE_DEVICES"] = visible_devices

    smoke_code = """
import torch

if not torch.cuda.is_available():
    print("CUDA_NOT_AVAILABLE")
    raise SystemExit(1)

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cap = torch.cuda.get_device_capability(i)
    print(f"GPU{i}={p.name};cap={cap}")

x = torch.randn(32, 32, device='cuda')
y = torch.randn(32, 32, device='cuda')
z = x @ y
_ = z.sum().item()
print("CUDA_SMOKE_OK")
""".strip()

    p = subprocess.run(
        ["python", "-c", smoke_code],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    out = (p.stdout or "").strip()
    ok = (p.returncode == 0) and ("CUDA_SMOKE_OK" in out)
    return ok, out


requested_visible = str(CFG.get("effective_cuda_visible_devices", CFG.get("cuda_visible_devices", "0")))
prefer_two = bool(CFG.get("use_two_gpus", True))

eval_visible = requested_visible
cnt = _probe_cuda_count_for_env(eval_visible)
if prefer_two and ("," in eval_visible) and cnt < 2:
    eval_visible = "0"
    cnt = _probe_cuda_count_for_env(eval_visible)

if cnt == 0:
    eval_visible = ""

env = os.environ.copy()
if eval_visible == "":
    env.pop("CUDA_VISIBLE_DEVICES", None)
else:
    env["CUDA_VISIBLE_DEVICES"] = eval_visible

print("CUDA_VISIBLE_DEVICES for eval:", repr(env.get("CUDA_VISIBLE_DEVICES", "<unset>")))
print("torch visible GPU count for eval:", cnt)
if cnt > 0:
    smoke_ok, smoke_log = _probe_cuda_smoke_for_env(eval_visible)
    print("cuda smoke test:", "ok" if smoke_ok else "failed")
    if smoke_log:
        print(smoke_log[:1200])
    if not smoke_ok:
        hint = ""
        if "no kernel image is available for execution on the device" in smoke_log.lower():
            hint = (
                "\nDetected CUDA kernel/image mismatch. "
                "Most likely torch-cuda build is incompatible with this GPU architecture. "
                "On Kaggle, switch Accelerator to T4, then restart session and run all cells again."
            )
        raise RuntimeError("CUDA preflight failed before eval." + hint)
proc = subprocess.run(cmd, cwd=repo_dir, env=env)
if proc.returncode != 0:
    raise RuntimeError(f"Evaluation failed with exit code {proc.returncode}")

print("Evaluation done.")



In [ ]:
# 7) Final metrics report (Train/Val/Test): Accuracy + Dice + IoU + compact .pt export
import sys
import json
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

repo_dir = Path(CFG["repo_dir"])
sys.path.insert(0, str(repo_dir))

import Config as config
from Load_Dataset import ValGenerator, ImageToImage2D
from nets.LViT import LViT
from utils import read_text

TASK_NAME_FOR_PATH = str(globals().get("TASK_NAME_CANONICAL", CFG.get("task_name", getattr(config, "task_name", "MosMedDataPlus"))))
MODEL_NAME_FOR_PATH = str(CFG.get("model_name", getattr(config, "model_name", "LViT")))
RUN_NAME_FOR_PATH = str(CFG.get("run_name", getattr(config, "session_name", "run")))

run_dir = repo_dir / TASK_NAME_FOR_PATH / MODEL_NAME_FOR_PATH / RUN_NAME_FOR_PATH
model_dir = run_dir / "models"
assert model_dir.exists(), f"Model directory not found: {model_dir}"

best_pth = model_dir / f"best_model-{MODEL_NAME_FOR_PATH}.pth.tar"
best_pt = model_dir / f"best_model-{MODEL_NAME_FOR_PATH}.pt"
last_pth = model_dir / "last_checkpoint.pth.tar"
last_pt = model_dir / "last_checkpoint.pt"

# Convert only key checkpoints to .pt (avoid doubling disk for every epoch)
def to_pt_if_needed(src_pth: Path, dst_pt: Path):
    if (not src_pth.exists()) or dst_pt.exists():
        return False
    obj = torch.load(src_pth, map_location="cpu")
    if isinstance(obj, dict):
        payload = {
            "epoch": obj.get("epoch"),
            "model": obj.get("model"),
            "state_dict": obj.get("state_dict", obj),
            "val_loss": obj.get("val_loss"),
        }
    else:
        payload = {"state_dict": obj}
    torch.save(payload, dst_pt)
    return True

conv_best = to_pt_if_needed(best_pth, best_pt)
conv_last = to_pt_if_needed(last_pth, last_pt)
print(f"Converted best->pt: {conv_best}, last->pt: {conv_last}")

# Pick best checkpoint for report
if best_pt.exists():
    ckpt_path = best_pt
elif best_pth.exists():
    ckpt_path = best_pth
elif last_pt.exists():
    ckpt_path = last_pt
elif last_pth.exists():
    ckpt_path = last_pth
else:
    cands = sorted(list(model_dir.glob("model-*.pt")) + list(model_dir.glob("model-*.pth.tar")))
    if not cands:
        raise FileNotFoundError(f"No checkpoint found in {model_dir}")
    ckpt_path = cands[-1]

print("Using checkpoint for report:", ckpt_path)

config_vit = config.get_CTranS_config()
model = LViT(config_vit, n_channels=config.n_channels, n_classes=config.n_labels)
obj = torch.load(ckpt_path, map_location="cpu")
state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj

try:
    model.load_state_dict(state, strict=True)
except Exception:
    stripped = {}
    for k, v in state.items():
        if str(k).startswith("module."):
            stripped[str(k)[7:]] = v
        else:
            stripped[str(k)] = v
    model.load_state_dict(stripped, strict=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

val_tf = ValGenerator(output_size=[config.img_size, config.img_size])


def _resolve_dataset_path(path_like):
    raw = Path(str(path_like))
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([
            repo_dir / raw,
            Path.cwd() / raw,
            raw,
        ])

    seen = set()
    dedup = []
    for p in candidates:
        rp = p.resolve()
        sp = str(rp)
        if sp not in seen:
            dedup.append(rp)
            seen.add(sp)

    for p in dedup:
        if p.exists():
            return p

    # default: repo-relative path even if missing, for cleaner error hints
    return (repo_dir / raw).resolve()


def _find_text_file(split_name: str, split_path: Path):
    split_name = split_name.lower().strip()
    task_root = split_path.parent

    preferred_names = {
        "train": ["Train_text.xlsx", "train_text.xlsx", "Train_Val_text.xlsx"],
        "val": ["Val_text.xlsx", "val_text.xlsx", "Train_text.xlsx", "train_text.xlsx", "Train_Val_text.xlsx"],
        "test": ["Test_text.xlsx", "test_text.xlsx", "Val_text.xlsx", "val_text.xlsx", "Train_text.xlsx", "train_text.xlsx", "Train_Val_text.xlsx"],
    }[split_name]

    candidates = []

    # First, try inside the split folder itself and task root.
    for nm in preferred_names:
        candidates.append(split_path / nm)
        candidates.append(task_root / nm)

    # Then, try canonical LViT locations.
    candidates.extend([
        task_root / "Train_Folder" / "Train_text.xlsx",
        task_root / "Val_Folder" / "Val_text.xlsx",
        task_root / "Test_Folder" / "Test_text.xlsx",
    ])

    seen = set()
    dedup = []
    for c in candidates:
        rc = c.resolve()
        sc = str(rc)
        if sc not in seen:
            dedup.append(rc)
            seen.add(sc)

    for c in dedup:
        if c.exists() and c.is_file():
            return c, dedup

    return None, dedup


def _build_default_text_map(split_path: Path):
    label_dir = split_path / "labelcol"
    if not label_dir.exists():
        raise FileNotFoundError(f"labelcol not found under split path: {split_path}")

    default_prompt = str(getattr(config, "default_text_prompt", "lung infection lesion"))
    names = sorted([p.name for p in label_dir.iterdir() if p.is_file()])
    if len(names) == 0:
        raise FileNotFoundError(f"No mask files found in {label_dir}")
    return {n: default_prompt for n in names}


def _build_loader(split_name: str):
    split_attr = f"{split_name}_dataset"
    split_cfg_value = getattr(config, split_attr)
    split_path = _resolve_dataset_path(split_cfg_value)

    if not split_path.exists():
        raise FileNotFoundError(
            f"Split path not found for '{split_name}': {split_cfg_value} -> {split_path}. "
            f"cwd={Path.cwd()} repo_dir={repo_dir}"
        )

    text_file, tried_files = _find_text_file(split_name, split_path)
    if text_file is not None:
        txt = read_text(text_file)
        text_source = str(text_file)
    else:
        txt = _build_default_text_map(split_path)
        text_source = "AUTO(default_text_prompt; no xlsx found)"

    ds = ImageToImage2D(str(split_path), config.task_name, txt, val_tf, image_size=config.img_size)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)
    return ds, loader, text_source, split_path


def _eval(loader):
    eps = 1e-7
    n = 0
    acc_sum = 0.0
    dice_sum = 0.0
    iou_sum = 0.0

    with torch.no_grad():
        for sampled_batch, _ in loader:
            images = sampled_batch["image"].to(device)
            masks = sampled_batch["label"].to(device)
            text = sampled_batch["text"].to(device)
            if text.shape[1] > 10:
                text = text[:, :10, :]

            preds = model(images, text)
            pred_bin = (preds > 0.5).float()
            mask_bin = (masks > 0).float()
            if mask_bin.ndim == 3:
                mask_bin = mask_bin.unsqueeze(1)

            inter = (pred_bin * mask_bin).sum().item()
            pred_area = pred_bin.sum().item()
            mask_area = mask_bin.sum().item()

            dice = (2.0 * inter + eps) / (pred_area + mask_area + eps)
            iou = (inter + eps) / (pred_area + mask_area - inter + eps)
            acc = (pred_bin == mask_bin).float().mean().item()

            acc_sum += float(acc)
            dice_sum += float(dice)
            iou_sum += float(iou)
            n += 1

    return {
        "samples": int(n),
        "accuracy": float(acc_sum / max(1, n)),
        "dice": float(dice_sum / max(1, n)),
        "iou": float(iou_sum / max(1, n)),
    }


rows = []
for split in ["train", "val", "test"]:
    ds, loader, text_file, split_path = _build_loader(split)
    print(f"[{split}] data={split_path} text={text_file}")
    m = _eval(loader)
    row = {"split": split, **m}
    rows.append(row)
    print(f"[{split}] samples={m['samples']} acc={m['accuracy']:.6f} dice={m['dice']:.6f} iou={m['iou']:.6f}")

metrics_df = pd.DataFrame(rows)
display(metrics_df)

metrics_csv = run_dir / "metrics_report.csv"
metrics_json = run_dir / "metrics_report.json"
metrics_df.to_csv(metrics_csv, index=False)
metrics_json.write_text(json.dumps(rows, indent=2), encoding="utf-8")

print("Saved:", metrics_csv)
print("Saved:", metrics_json)



In [ ]:
# 7) Quick check outputs
from pathlib import Path

repo_dir = Path(CFG["repo_dir"])
run_dir = repo_dir / TASK_NAME_CANONICAL / CFG["model_name"] / CFG["run_name"]

print("Run dir:", run_dir)
print("Model dir:", run_dir / "models")
print("Log file:", run_dir / f"{CFG['run_name']}.log")
print("Visualize val dir:", run_dir / "visualize_val")

if (run_dir / "models").exists():
    print("Model files:")
    for p in sorted((run_dir / "models").glob("*.pth.tar"))[:20]:
        print("-", p.name)

metrics_csv = run_dir / 'metrics_report.csv'
metrics_json = run_dir / 'metrics_report.json'
print('Metrics CSV:', metrics_csv, 'exists=', metrics_csv.exists())
print('Metrics JSON:', metrics_json, 'exists=', metrics_json.exists())

pt_files = sorted((run_dir / 'models').glob('*.pt')) if (run_dir / 'models').exists() else []
if pt_files:
    print('PT model files:')
    for p in pt_files[:30]:
        print('-', p.name)

epoch_csv = run_dir / 'epoch_training_metrics.csv'
epoch_json = run_dir / 'epoch_training_metrics.json'
print('Epoch Metrics CSV:', epoch_csv, 'exists=', epoch_csv.exists())
print('Epoch Metrics JSON:', epoch_json, 'exists=', epoch_json.exists())

pseudo_csv = run_dir / 'pseudo_label_vs_ground_truth_holdout.csv'
pseudo_json = run_dir / 'pseudo_label_vs_ground_truth_holdout.json'
print('Pseudo CSV:', pseudo_csv, 'exists=', pseudo_csv.exists())
print('Pseudo JSON:', pseudo_json, 'exists=', pseudo_json.exists())


In [ ]:

# 8) Pseudo-label quality report (held-out train labels): CSV + JSON
import sys
import json
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

repo_dir = Path(CFG["repo_dir"])
sys.path.insert(0, str(repo_dir))

import Config as config
from Load_Dataset import ValGenerator, ImageToImage2D
from nets.LViT import LViT
from utils import read_text

TASK_NAME_FOR_PATH = str(globals().get("TASK_NAME_CANONICAL", CFG.get("task_name", getattr(config, "task_name", "QaTa-Covid19"))))
MODEL_NAME_FOR_PATH = str(CFG.get("model_name", getattr(config, "model_name", "LViT")))
RUN_NAME_FOR_PATH = str(CFG.get("run_name", getattr(config, "session_name", "run")))

run_dir = repo_dir / TASK_NAME_FOR_PATH / MODEL_NAME_FOR_PATH / RUN_NAME_FOR_PATH
model_dir = run_dir / "models"
assert model_dir.exists(), f"Model directory not found: {model_dir}"

best_pth = model_dir / f"best_model-{MODEL_NAME_FOR_PATH}.pth.tar"
best_pt = model_dir / f"best_model-{MODEL_NAME_FOR_PATH}.pt"
last_pth = model_dir / "last_checkpoint.pth.tar"
last_pt = model_dir / "last_checkpoint.pt"

if best_pt.exists():
    ckpt_path = best_pt
elif best_pth.exists():
    ckpt_path = best_pth
elif last_pt.exists():
    ckpt_path = last_pt
elif last_pth.exists():
    ckpt_path = last_pth
else:
    cands = sorted(list(model_dir.glob("model-*.pt")) + list(model_dir.glob("model-*.pth.tar")))
    if not cands:
        raise FileNotFoundError(f"No checkpoint found in {model_dir}")
    ckpt_path = cands[-1]

print("Using checkpoint for pseudo-label report:", ckpt_path)

config_vit = config.get_CTranS_config()
model = LViT(config_vit, n_channels=config.n_channels, n_classes=config.n_labels)
obj = torch.load(ckpt_path, map_location="cpu")
state = obj.get("state_dict", obj) if isinstance(obj, dict) else obj

try:
    model.load_state_dict(state, strict=True)
except Exception:
    stripped = {}
    for k, v in state.items():
        k2 = str(k)
        if k2.startswith("module."):
            stripped[k2[7:]] = v
        else:
            stripped[k2] = v
    model.load_state_dict(stripped, strict=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

val_tf = ValGenerator(output_size=[config.img_size, config.img_size])


def _resolve_dataset_path(path_like):
    raw = Path(str(path_like))
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([
            repo_dir / raw,
            Path.cwd() / raw,
            raw,
        ])

    seen = set()
    dedup = []
    for p in candidates:
        rp = p.resolve()
        sp = str(rp)
        if sp not in seen:
            dedup.append(rp)
            seen.add(sp)

    for p in dedup:
        if p.exists():
            return p

    return (repo_dir / raw).resolve()


def _build_default_text_map(split_path: Path):
    label_dir = split_path / "labelcol"
    if not label_dir.exists():
        raise FileNotFoundError(f"labelcol not found under split path: {split_path}")

    default_prompt = str(getattr(config, "default_text_prompt", "lung infection lesion"))
    names = sorted([p.name for p in label_dir.iterdir() if p.is_file()])
    if len(names) == 0:
        raise FileNotFoundError(f"No mask files found in {label_dir}")
    return {n: default_prompt for n in names}


def _load_train_text(split_path: Path):
    task_root = split_path.parent
    candidates = [
        split_path / "Train_text.xlsx",
        split_path / "train_text.xlsx",
        task_root / "Train_Folder" / "Train_text.xlsx",
        task_root / "Train_text.xlsx",
        task_root / "Train_Val_text.xlsx",
    ]
    seen = set()
    for c in candidates:
        rc = c.resolve()
        sc = str(rc)
        if sc in seen:
            continue
        seen.add(sc)
        if rc.exists() and rc.is_file():
            return read_text(rc), str(rc)

    return _build_default_text_map(split_path), "AUTO(default_text_prompt; no xlsx found)"


train_split_path = _resolve_dataset_path(getattr(config, "train_dataset"))
if not train_split_path.exists():
    raise FileNotFoundError(f"Train split path not found: {train_split_path}")

train_text, train_text_source = _load_train_text(train_split_path)
full_train_ds = ImageToImage2D(str(train_split_path), config.task_name, train_text, val_tf, image_size=config.img_size)

train_subset_ratio = float(getattr(config, "train_subset_ratio", 1.0))
if train_subset_ratio <= 0.0:
    raise ValueError(f"train_subset_ratio must be > 0, got {train_subset_ratio}")

full_train_len = len(full_train_ds)
subset_len = max(1, int(round(full_train_len * train_subset_ratio)))
subset_len = min(full_train_len, subset_len)

subset_generator = torch.Generator()
subset_generator.manual_seed(int(getattr(config, "seed", 666)))
perm = torch.randperm(full_train_len, generator=subset_generator).tolist()
train_indices = perm[:subset_len]
holdout_indices = perm[subset_len:]

if len(holdout_indices) == 0:
    raise ValueError(
        f"No holdout samples left for pseudo-label evaluation. train_subset_ratio={train_subset_ratio}, total={full_train_len}"
    )

holdout_ds = Subset(full_train_ds, holdout_indices)
holdout_loader = DataLoader(holdout_ds, batch_size=1, shuffle=False, num_workers=0, pin_memory=False)

print(f"train_text source: {train_text_source}")
print(f"Pseudo-label holdout samples: {len(holdout_indices)}/{full_train_len} (ratio={1.0 - train_subset_ratio:.0%})")

eps = 1e-7
n = 0

tp_sum = 0.0
fp_sum = 0.0
fn_sum = 0.0
tn_sum = 0.0

sample_acc_sum = 0.0
sample_precision_sum = 0.0
sample_recall_sum = 0.0
sample_dice_sum = 0.0
sample_iou_sum = 0.0

with torch.no_grad():
    for sampled_batch, _ in holdout_loader:
        images = sampled_batch["image"].to(device)
        masks = sampled_batch["label"].to(device)
        text = sampled_batch["text"].to(device)

        if text.shape[1] > 10:
            text = text[:, :10, :]

        preds = model(images, text)
        pred_bin = (preds > 0.5).float()
        mask_bin = (masks > 0).float()

        if pred_bin.ndim == 3:
            pred_bin = pred_bin.unsqueeze(1)
        if mask_bin.ndim == 3:
            mask_bin = mask_bin.unsqueeze(1)

        tp = (pred_bin * mask_bin).sum().item()
        fp = (pred_bin * (1.0 - mask_bin)).sum().item()
        fn = ((1.0 - pred_bin) * mask_bin).sum().item()
        tn = ((1.0 - pred_bin) * (1.0 - mask_bin)).sum().item()

        tp_sum += float(tp)
        fp_sum += float(fp)
        fn_sum += float(fn)
        tn_sum += float(tn)

        sample_precision = (tp + eps) / (tp + fp + eps)
        sample_recall = (tp + eps) / (tp + fn + eps)
        sample_dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
        sample_iou = (tp + eps) / (tp + fp + fn + eps)
        sample_acc = (tp + tn + eps) / (tp + fp + fn + tn + eps)

        sample_acc_sum += float(sample_acc)
        sample_precision_sum += float(sample_precision)
        sample_recall_sum += float(sample_recall)
        sample_dice_sum += float(sample_dice)
        sample_iou_sum += float(sample_iou)
        n += 1

pixel_precision = (tp_sum + eps) / (tp_sum + fp_sum + eps)
pixel_recall = (tp_sum + eps) / (tp_sum + fn_sum + eps)
pixel_dice = (2.0 * tp_sum + eps) / (2.0 * tp_sum + fp_sum + fn_sum + eps)
pixel_iou = (tp_sum + eps) / (tp_sum + fp_sum + fn_sum + eps)
pixel_accuracy = (tp_sum + tn_sum + eps) / (tp_sum + fp_sum + fn_sum + tn_sum + eps)

row = {
    "task": str(config.task_name),
    "run_name": RUN_NAME_FOR_PATH,
    "train_subset_ratio": float(train_subset_ratio),
    "total_train_samples": int(full_train_len),
    "labeled_train_samples": int(len(train_indices)),
    "pseudo_eval_holdout_samples": int(len(holdout_indices)),
    "checkpoint": str(ckpt_path),
    "pixel_accuracy": float(pixel_accuracy),
    "pixel_precision": float(pixel_precision),
    "pixel_recall": float(pixel_recall),
    "pixel_dice": float(pixel_dice),
    "pixel_iou": float(pixel_iou),
    "mean_sample_accuracy": float(sample_acc_sum / max(1, n)),
    "mean_sample_precision": float(sample_precision_sum / max(1, n)),
    "mean_sample_recall": float(sample_recall_sum / max(1, n)),
    "mean_sample_dice": float(sample_dice_sum / max(1, n)),
    "mean_sample_iou": float(sample_iou_sum / max(1, n)),
    "tp_pixels": float(tp_sum),
    "fp_pixels": float(fp_sum),
    "fn_pixels": float(fn_sum),
    "tn_pixels": float(tn_sum),
}

report_df = pd.DataFrame([row])
display(report_df)

pseudo_csv = run_dir / "pseudo_label_vs_ground_truth_holdout.csv"
pseudo_json = run_dir / "pseudo_label_vs_ground_truth_holdout.json"

report_df.to_csv(pseudo_csv, index=False)
pseudo_json.write_text(json.dumps([row], indent=2), encoding="utf-8")

print("Saved:", pseudo_csv)
print("Saved:", pseudo_json)


In [ ]:

# 11) Zip full run outputs (run this last)
import shutil
from pathlib import Path

repo_dir = Path(CFG["repo_dir"])
run_dir = repo_dir / TASK_NAME_CANONICAL / CFG["model_name"] / CFG["run_name"]
assert run_dir.exists(), f"Run directory not found: {run_dir}"

zip_base = Path('/kaggle/working') / f"output_{TASK_NAME_CANONICAL}_{CFG['model_name']}_{CFG['run_name']}_FULL"
zip_path = zip_base.with_suffix('.zip')
if zip_path.exists():
    zip_path.unlink()

created = shutil.make_archive(str(zip_base), 'zip', root_dir=str(run_dir.parent), base_dir=run_dir.name)

size_mb = Path(created).stat().st_size / (1024 * 1024)
print('Created full ZIP:', created)
print(f'ZIP size: {size_mb:.2f} MB')
print('Included run dir:', run_dir)

# Quick artifact existence summary
must_have = [
    run_dir / 'metrics_report.csv',
    run_dir / 'metrics_report.json',
    run_dir / 'pseudo_label_vs_ground_truth_holdout.csv',
    run_dir / 'pseudo_label_vs_ground_truth_holdout.json',
    run_dir / 'epoch_training_metrics.csv',
    run_dir / 'epoch_training_metrics.json',
]
for p in must_have:
    print(f'{p.name}:', p.exists())
